In [22]:
import pandas as pd
import numpy as np

np.random.seed(42)

# 1. Create Menu Dataset
menu_data = (
    {"item_id": 101, "item_name": "Paneer Tikka", "category": "Starters", "price": 280, "cost": 95},
    {"item_id": 102, "item_name": "Crispy Corn", "category": "Starters", "price": 220, "cost": 60},
    {"item_id": 103, "item_name": "Chicken Wings", "category": "Starters", "price": 320, "cost": 140},
    {"item_id": 104, "item_name": "Butter Chicken", "category": "Main Course", "price": 420, "cost": 160},
    {"item_id": 105, "item_name": "Dal Makhani", "category": "Main Course", "price": 340, "cost": 85},
    {"item_id": 106, "item_name": "Veg Biryani", "category": "Main Course", "price": 310, "cost": 90},
    {"item_id": 107, "item_name": "Garlic Naan", "category": "Breads", "price": 80, "cost": 18},
    {"item_id": 108, "item_name": "Masala Chaas", "category": "Beverages", "price": 90, "cost": 20},
    {"item_id": 109, "item_name": "Cold Coffee", "category": "Beverages", "price": 160, "cost": 45},
    {"item_id": 110, "item_name": "Gulab Jamun", "category": "Desserts", "price": 130, "cost": 35}
)
menu_df = pd.DataFrame(menu_data)
menu_df["unit_profit"] = menu_df["price"] - menu_df["cost"]

# 2. Generate Synthetic Orders Dataset (1,500 order line-items)
n_rows = 1500
order_ids = np.random.randint(1001, 1650, size=n_rows)
weights = (0.12, 0.08, 0.09, 0.16, 0.14, 0.06, 0.18, 0.07, 0.05, 0.05)
item_ids = np.random.choice(menu_df["item_id"], size=n_rows, p=weights)

hour_options = (12, 13, 14, 16, 18, 19, 20, 21, 22)
hour_probs = (0.12, 0.18, 0.10, 0.04, 0.06, 0.14, 0.18, 0.12, 0.06)
hours = np.random.choice(hour_options, size=n_rows, p=hour_probs)

qty_options = (1, 2, 3)
qty_probs = (0.75, 0.20, 0.05)

orders_df = pd.DataFrame({
    "order_id": order_ids,
    "item_id": item_ids,
    "quantity": np.random.choice(qty_options, size=n_rows, p=qty_probs),
    "order_hour": hours
})

print("Menu shape:", menu_df.shape)
print("Orders shape:", orders_df.shape)
orders_df.head()


Menu shape: (10, 6)
Orders shape: (1500, 4)


,order_id,item_id,quantity,order_hour
0,1103,104,1,20
1,1436,105,1,19
2,1271,102,1,14
3,1107,107,1,14
4,1072,107,1,12


In [23]:
# Merge orders with menu details
df = orders_df.merge(menu_df, on="item_id", how="left")

# Check missing values and remove duplicates
print("Missing values per column:\n", df.isnull().sum(), "\n")
df = df.drop_duplicates()

# Calculate revenue, cost, and profit per row
df["total_revenue"] = df["price"] * df["quantity"]
df["total_cost"] = df["cost"] * df["quantity"]
df["total_profit"] = df["unit_profit"] * df["quantity"]

# Overall KPIs using NumPy & Pandas
total_orders = df["order_id"].nunique()
total_items = np.sum(df["quantity"])
total_rev = np.sum(df["total_revenue"])
total_prof = np.sum(df["total_profit"])
avg_order_val = total_rev / total_orders

print("--- Overall Restaurant KPIs ---")
print(f"Total Unique Orders : {total_orders}")
print(f"Total Items Sold    : {total_items}")
print(f"Total Revenue       : ₹{total_rev:,.0f}")
print(f"Total Gross Profit  : ₹{total_prof:,.0f}")
print(f"Average Order Value : ₹{avg_order_val:,.2f}")


Missing values per column:
 order_id       0
item_id        0
quantity       0
order_hour     0
item_name      0
category       0
price          0
cost           0
unit_profit    0
dtype: int64 

--- Overall Restaurant KPIs ---
Total Unique Orders : 575
Total Items Sold    : 1919
Total Revenue       : ₹472,540
Total Gross Profit  : ₹319,870
Average Order Value : ₹821.81


In [24]:
group_cols = list(("item_name", "category", "price", "unit_profit"))

item_perf = df.groupby(group_cols, as_index=False).agg(
    units_sold=("quantity", "sum"),
    total_revenue=("total_revenue", "sum"),
    total_profit=("total_profit", "sum")
)

# Calculate benchmarks and contribution shares with NumPy
avg_units = np.mean(item_perf["units_sold"])
avg_margin = np.mean(item_perf["unit_profit"])
item_perf["profit_share_%"] = np.round((item_perf["total_profit"] / np.sum(item_perf["total_profit"])) * 100, 2)

# Vectorized Quadrant Classification with np.select
high_pop = item_perf["units_sold"] >= avg_units
high_prof = item_perf["unit_profit"] >= avg_margin

conditions = (
    high_pop & high_prof,
    high_pop & (~high_prof),
    (~high_pop) & high_prof
)
choices = ("Star", "Plow Horse", "Puzzle")
item_perf["quadrant"] = np.select(conditions, choices, default="Dog")

# Display sorted Menu Engineering table
item_perf.sort_values(by="total_profit", ascending=False)


,item_name,category,price,unit_profit,units_sold,total_revenue,total_profit,profit_share_%,quadrant
0,Butter Chicken,Main Course,420,260,308,129360,80080,25.04,Star
4,Dal Makhani,Main Course,340,255,254,86360,64770,20.25,Star
8,Paneer Tikka,Starters,280,185,242,67760,44770,14.00,Star
1,Chicken Wings,Starters,320,180,162,51840,29160,9.12,Puzzle
9,Veg Biryani,Main Course,310,220,128,39680,28160,8.80,Puzzle
5,Garlic Naan,Breads,80,62,370,29600,22940,7.17,Plow Horse
3,Crispy Corn,Starters,220,160,135,29700,21600,6.75,Dog
6,Gulab Jamun,Desserts,130,95,110,14300,10450,3.27,Dog
7,Masala Chaas,Beverages,90,70,138,12420,9660,3.02,Dog
2,Cold Coffee,Beverages,160,115,72,11520,8280,2.59,Dog


In [25]:
# 1. Category Performance Summary
cat_summary = df.groupby("category", as_index=False).agg(
    items_sold=("quantity", "sum"),
    total_revenue=("total_revenue", "sum"),
    total_profit=("total_profit", "sum")
)
cat_summary["profit_margin_%"] = np.round((cat_summary["total_profit"] / cat_summary["total_revenue"]) * 100, 2)
cat_summary = cat_summary.sort_values(by="total_revenue", ascending=False)

print("--- Category Performance ---")
print(cat_summary.to_string(index=False), "\n")

# 2. Peak Dining Hours Summary
hourly_summary = df.groupby("order_hour", as_index=False).agg(
    unique_orders=("order_id", "nunique"),
    items_sold=("quantity", "sum"),
    revenue=("total_revenue", "sum")
)
hourly_summary["demand_level"] = np.where(
    hourly_summary["unique_orders"] >= np.percentile(hourly_summary["unique_orders"], 75),
    "Peak Rush",
    "Normal"
)

print("--- Hourly Demand Breakdown ---")
print(hourly_summary.to_string(index=False), "\n")

# 3. Quadrant Summary
quadrant_summary = item_perf.groupby("quadrant", as_index=False).agg(
    dish_count=("item_name", "count"),
    total_units=("units_sold", "sum"),
    total_profit=("total_profit", "sum")
).sort_values(by="total_profit", ascending=False)

print("--- Menu Quadrant Totals ---")
print(quadrant_summary.to_string(index=False))


--- Category Performance ---
   category  items_sold  total_revenue  total_profit  profit_margin_%
Main Course         690         255400        173010            67.74
   Starters         539         149300         95530            63.99
     Breads         370          29600         22940            77.50
  Beverages         210          23940         17940            74.94
   Desserts         110          14300         10450            73.08 

--- Hourly Demand Breakdown ---
 order_hour  unique_orders  items_sold  revenue demand_level
         12            160         238    58500       Normal
         13            220         350    83470    Peak Rush
         14            137         188    46590       Normal
         16             62          81    18850       Normal
         18             69          94    23150       Normal
         19            171         276    68700    Peak Rush
         20            218         338    85220    Peak Rush
         21            156   